In [1]:
import numpy as np

class Layer:
    def forward(self, inputs):
        raise NotImplementedError
    
    def backward(self, grad_output, learning_rate):
        raise NotImplementedError

class Dense(Layer):
    def __init__(self, input_dim, output_dim):
        self.weights = np.random.randn(input_dim, output_dim) * 0.01
        self.bias = np.zeros((1, output_dim))
    
    def forward(self, inputs):
        self.inputs = inputs
        return np.dot(inputs, self.weights) + self.bias
    
    def backward(self, grad_output, learning_rate):
        grad_input = np.dot(grad_output, self.weights.T)
        grad_weights = np.dot(self.inputs.T, grad_output)
        grad_bias = np.sum(grad_output, axis=0, keepdims=True)
        
        # Update parameters
        self.weights -= learning_rate * grad_weights
        self.bias -= learning_rate * grad_bias
        
        return grad_input

class Activation(Layer):
    def forward(self, inputs):
        raise NotImplementedError
    
    def backward(self, grad_output, learning_rate):
        raise NotImplementedError

class ReLU(Activation):
    def forward(self, inputs):
        self.inputs = inputs
        return np.maximum(0, inputs)
    
    def backward(self, grad_output, learning_rate):
        grad_input = grad_output * (self.inputs > 0)
        return grad_input

class Sigmoid(Activation):
    def forward(self, inputs):
        self.outputs = 1 / (1 + np.exp(-inputs))
        return self.outputs
    
    def backward(self, grad_output, learning_rate):
        return grad_output * (self.outputs * (1 - self.outputs))

class Loss:
    def loss(self, y_pred, y_true):
        raise NotImplementedError
    
    def grad(self, y_pred, y_true):
        raise NotImplementedError

class MSELoss(Loss):
    def loss(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)
    
    def grad(self, y_pred, y_true):
        return 2 * (y_pred - y_true) / y_true.size

class NeuralNetwork:
    def __init__(self):
        self.layers = []
    
    def add(self, layer):
        self.layers.append(layer)
    
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, grad_output, learning_rate):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)
    
    def accuracy(self, y_pred, y_true):
        # For binary classification
        y_pred_labels = (y_pred > 0.5).astype(int)
        correct = np.sum(y_pred_labels == y_true)
        return correct / y_true.shape[0]
    
    def train(self, X, y, loss_fn, epochs=1000, learning_rate=0.01):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            
            loss = loss_fn.loss(y_pred, y)
            
            grad = loss_fn.grad(y_pred, y)
            self.backward(grad, learning_rate)
            
            acc = self.accuracy(y_pred, y)
            
            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Accuracy: {acc:.2f}")

if __name__ == "__main__":
    X = np.array([[0,0],[0,1],[1,0],[1,1]])
    y = np.array([[0],[1],[1],[0]])  # XOR output
    
    model = NeuralNetwork()
    model.add(Dense(2, 4))
    model.add(ReLU())
    model.add(Dense(4, 1))
    model.add(Sigmoid())
    
    loss_fn = MSELoss()
    model.train(X, y, loss_fn, epochs=1000, learning_rate=0.1)
    
    print("\nFinal Predictions:")
    preds = model.forward(X)
    print(preds)
    print("Final Accuracy:", model.accuracy(preds, y))


Epoch 100, Loss: 0.2500, Accuracy: 0.50
Epoch 200, Loss: 0.2500, Accuracy: 0.75
Epoch 300, Loss: 0.2499, Accuracy: 0.75
Epoch 400, Loss: 0.2497, Accuracy: 0.75
Epoch 500, Loss: 0.2494, Accuracy: 0.75
Epoch 600, Loss: 0.2485, Accuracy: 0.75
Epoch 700, Loss: 0.2465, Accuracy: 0.75
Epoch 800, Loss: 0.2417, Accuracy: 0.75
Epoch 900, Loss: 0.2318, Accuracy: 0.75
Epoch 1000, Loss: 0.2134, Accuracy: 0.75

Final Predictions:
[[0.47297797]
 [0.60772706]
 [0.49927557]
 [0.47333234]]
Final Accuracy: 0.75
